In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os

# --- 1️⃣ Đọc dữ liệu gốc ---
df_train = pd.read_csv('../data/01_raw/train.csv')
df_test = pd.read_csv('../data/01_raw/test.csv')

# --- 2️⃣ Điền các giá trị thiếu ---
cols_none = [
    'Alley', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType'
]
df_train[cols_none] = df_train[cols_none].fillna('None')
df_test[cols_none] = df_test[cols_none].fillna('None')

for col in ['MasVnrArea', 'GarageYrBlt']:
    df_train[col] = df_train[col].fillna(0)
    df_test[col] = df_test[col].fillna(0)

df_train['Electrical'] = df_train['Electrical'].fillna(df_train['Electrical'].mode()[0])
df_test['Electrical'] = df_test['Electrical'].fillna(df_test['Electrical'].mode()[0])

df_train['LotFrontage'] = df_train.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
df_test['LotFrontage'] = df_test.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

# --- 3️⃣ One-hot encoding ---
df_train = pd.get_dummies(df_train, drop_first=True)
df_test = pd.get_dummies(df_test, drop_first=True)

# --- 4️⃣ Đồng bộ cột ---
df_test = df_test.reindex(columns=[col for col in df_train.columns if col != 'SalePrice'], fill_value=0)

# --- 5️⃣ Feature Engineering ---
df_train['TotalSF'] = df_train['TotalBsmtSF'] + df_train['1stFlrSF'] + df_train['2ndFlrSF']
df_train['TotalBathrooms'] = df_train['FullBath'] + 0.5 * df_train['HalfBath'] + \
                             df_train['BsmtFullBath'] + 0.5 * df_train['BsmtHalfBath']
df_train['Age'] = df_train['YrSold'] - df_train['YearBuilt']

df_test['TotalSF'] = df_test['TotalBsmtSF'] + df_test['1stFlrSF'] + df_test['2ndFlrSF']
df_test['TotalBathrooms'] = df_test['FullBath'] + 0.5 * df_test['HalfBath'] + \
                            df_test['BsmtFullBath'] + 0.5 * df_test['BsmtHalfBath']
df_test['Age'] = df_test['YrSold'] - df_test['YearBuilt']

# --- 6️⃣ Log-transform SalePrice ---
df_train['SalePrice'] = np.log1p(df_train['SalePrice'])

# --- 7️⃣ Chuẩn hóa các đặc trưng số ---
num_features = df_train.select_dtypes(include=[np.number]).columns.drop('SalePrice')

scaler = StandardScaler()
df_train[num_features] = scaler.fit_transform(df_train[num_features])
df_test[num_features] = scaler.transform(df_test[num_features])

# --- 8️⃣ Lưu dữ liệu ---
os.makedirs("../data/03_processed", exist_ok=True)
df_train.to_csv("../data/03_processed/train_clean.csv", index=False)
df_test.to_csv("../data/03_processed/test_clean.csv", index=False)

print("✅ Dữ liệu đã sẵn sàng cho mô hình (03_processed).")


✅ Dữ liệu đã sẵn sàng cho mô hình (03_processed).
